# Predicting the Annual Turnover of a Restaurant  (CatBoost)
---

Goal:
- Train on Train_dataset_.csv
- Predict Annual Turnover for Test_dataset_.csv
- Submit CSV with:
    1) Registration Number
    2) Annual Turnover

Metric:
- RMSE (lower is better)

Why CatBoost?
- Handles categorical columns well (no heavy one-hot encoding needed)

Key trick:
- Turnover is usually skewed, so we train on log1p(target) and convert back with expm1.

---

## 2. CAT Boost with oof

In [9]:
"""
Restaurant Turnover Prediction — Strong CatBoost Pipeline
--------------------------------------------------------
Adds:
1) Frequency Encoding for categorical columns
2) OOF Target Encoding (safe) using log1p(Annual Turnover)
3) 6-model CatBoost ensemble (average predictions in log space)

Output:
submission_catboost_ensemble_oof.csv
Columns: Registration Number, Annual Turnover
"""

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostRegressor


# =========================
# 1) SETTINGS
# =========================
TRAIN_PATH = "Train_dataset_.csv"
TEST_PATH  = "Test_dataset_.csv"

ID_COL     = "Registration Number"
TARGET_COL = "Annual Turnover"

# OOF settings
N_SPLITS = 5
N_BINS   = 10
SEED     = 42

# If a categorical column is almost unique per row, it behaves like an ID and can overfit.
# We exclude such columns from OOF/frequency encoding automatically.
MAX_UNIQUE_RATIO = 0.50   # exclude if nunique > 50% of rows


# =========================
# 2) LOAD DATA
# =========================
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

# Basic checks
for col in [ID_COL, TARGET_COL]:
    if col not in train_df.columns:
        raise ValueError(f"Missing '{col}' in train.")
if ID_COL not in test_df.columns:
    raise ValueError(f"Missing '{ID_COL}' in test.")


# =========================
# 3) SPLIT X/y (DROP ID FROM FEATURES)
# =========================
y = train_df[TARGET_COL].copy()
y_log = np.log1p(y)

X = train_df.drop(columns=[TARGET_COL, ID_COL]).copy()
X_test = test_df.drop(columns=[ID_COL]).copy()

# Align test columns to train columns (safety)
X_test = X_test.reindex(columns=X.columns, fill_value=np.nan)


# =========================
# 4) IDENTIFY CATEGORICAL & NUMERICAL
# =========================
cat_cols_all = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols_all]

# Cast categoricals to string early (consistent mapping)
for c in cat_cols_all:
    X[c] = X[c].astype(str)
    X_test[c] = X_test[c].astype(str)

# Fill missing values (simple & consistent)
# - categoricals: "Unknown"
# - numerics: median from train
for c in cat_cols_all:
    X[c] = X[c].replace({"nan": "Unknown"}).fillna("Unknown")
    X_test[c] = X_test[c].replace({"nan": "Unknown"}).fillna("Unknown")

for c in num_cols:
    med = pd.to_numeric(X[c], errors="coerce").median()
    X[c] = pd.to_numeric(X[c], errors="coerce").fillna(med)
    X_test[c] = pd.to_numeric(X_test[c], errors="coerce").fillna(med)


# =========================
# 5) SELECT WHICH CATEGORICAL COLS TO ENCODE (avoid near-IDs)
# =========================
n_rows = len(X)
cat_cols = []
for c in cat_cols_all:
    uniq_ratio = X[c].nunique(dropna=False) / n_rows
    if uniq_ratio <= MAX_UNIQUE_RATIO:
        cat_cols.append(c)

print(f"Categorical columns total: {len(cat_cols_all)}")
print(f"Categorical columns encoded (after unique filter): {len(cat_cols)}")


# =========================
# 6) FREQUENCY ENCODING (train stats applied to both train & test)
# =========================
def add_frequency_encoding(X_df: pd.DataFrame, X_test_df: pd.DataFrame, cols):
    X_df = X_df.copy()
    X_test_df = X_test_df.copy()

    for c in cols:
        freq = X_df[c].value_counts(dropna=False)
        # normalize: frequency proportion
        freq = freq / len(X_df)

        X_df[f"{c}__freq"] = X_df[c].map(freq).fillna(0).astype("float32")
        X_test_df[f"{c}__freq"] = X_test_df[c].map(freq).fillna(0).astype("float32")

    return X_df, X_test_df

X_fe, X_test_fe = add_frequency_encoding(X, X_test, cat_cols)


# =========================
# 7) SAFE OOF TARGET ENCODING (log-target)
# =========================
def make_bins(y_series: pd.Series, n_bins=10):
    """Stratification bins for regression folds."""
    return pd.qcut(y_series, q=n_bins, duplicates="drop").astype(str)

def add_oof_target_encoding(
    X_df: pd.DataFrame,
    X_test_df: pd.DataFrame,
    y_log: pd.Series,
    cols,
    n_splits=5,
    n_bins=10,
    seed=42
):
    """
    Adds OOF mean target encoding for each categorical column.
    - Train: encoding computed out-of-fold (safe)
    - Test: encoding computed from full train
    """
    X_df = X_df.copy()
    X_test_df = X_test_df.copy()

    # Prepare folds
    bins = make_bins(np.expm1(y_log), n_bins=n_bins)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    global_mean = float(y_log.mean())

    # Create empty columns
    for c in cols:
        X_df[f"{c}__oof_te"] = np.nan

    # OOF fill
    for tr_idx, val_idx in skf.split(X_df, bins):
        X_tr, X_val = X_df.iloc[tr_idx], X_df.iloc[val_idx]
        y_tr = y_log.iloc[tr_idx]

        # For each col, compute mean log-target by category on train fold only
        for c in cols:
            mapping = y_tr.groupby(X_tr[c]).mean()
            X_df.loc[X_df.index[val_idx], f"{c}__oof_te"] = X_val[c].map(mapping)

    # Fill any missing (unseen categories in fold) with global mean
    for c in cols:
        X_df[f"{c}__oof_te"] = X_df[f"{c}__oof_te"].fillna(global_mean).astype("float32")

        # Test encoding uses full train mapping
        full_mapping = y_log.groupby(X_df[c]).mean()
        X_test_df[f"{c}__oof_te"] = X_test_df[c].map(full_mapping).fillna(global_mean).astype("float32")

    return X_df, X_test_df

X_fe, X_test_fe = add_oof_target_encoding(
    X_fe, X_test_fe, y_log, cat_cols,
    n_splits=N_SPLITS, n_bins=N_BINS, seed=SEED
)


# =========================
# 8) CATBOOST SETUP
# =========================
# We keep original categorical columns + new numeric encodings.
# CatBoost will use categorical columns natively, and also benefit from the encoded numeric features.
cat_features_idx = [X_fe.columns.get_loc(c) for c in cat_cols_all]  # ALL original categoricals

print("Final feature count:", X_fe.shape[1])
print("CatBoost categorical features:", len(cat_features_idx))


# =========================
# 9) 6-MODEL CATBOOST ENSEMBLE (average in LOG space)
# =========================
models = [
    # shallower + stronger reg
    dict(iterations=3500, learning_rate=0.05, depth=6,  l2_leaf_reg=8, subsample=0.80, random_seed=SEED),
    dict(iterations=3500, learning_rate=0.05, depth=6,  l2_leaf_reg=10, subsample=0.85, random_seed=SEED+1),

    # medium
    dict(iterations=4500, learning_rate=0.04, depth=8,  l2_leaf_reg=6, subsample=0.80, random_seed=SEED+2),
    dict(iterations=4500, learning_rate=0.04, depth=8,  l2_leaf_reg=5, subsample=0.90, random_seed=SEED+3),

    # deeper (but controlled)
    dict(iterations=5500, learning_rate=0.035, depth=10, l2_leaf_reg=8, subsample=0.85, random_seed=SEED+4),
    dict(iterations=5500, learning_rate=0.035, depth=10, l2_leaf_reg=7, subsample=0.90, random_seed=SEED+5),
]

preds_log = []

for i, p in enumerate(models, 1):
    print(f"\n--- Training model {i}/6: depth={p['depth']} l2={p['l2_leaf_reg']} subsample={p['subsample']} seed={p['random_seed']} ---")

    m = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        verbose=200,
        allow_writing_files=False,
        **p
    )

    m.fit(X_fe, y_log, cat_features=cat_features_idx)
    preds_log.append(m.predict(X_test_fe))

# Average predictions in log space
avg_pred_log = np.mean(np.vstack(preds_log), axis=0)

# Convert back to original scale
pred = np.expm1(avg_pred_log)
pred = np.clip(pred, 0, None)


# =========================
# 10) SUBMISSION
# =========================
submission = pd.DataFrame({
    ID_COL: test_df[ID_COL].values,
    TARGET_COL: pred
})

submission.to_csv("submission_catboost_ensemble_oof.csv", index=False)
print("\n✔ Saved: submission_catboost_ensemble_oof.csv")
print(submission.head())



Train shape: (3493, 34)
Test shape : (500, 33)
Categorical columns total: 7
Categorical columns encoded (after unique filter): 6
Final feature count: 44
CatBoost categorical features: 7

--- Training model 1/6: depth=6 l2=8 subsample=0.8 seed=42 ---
0:	learn: 0.5425541	total: 38.3ms	remaining: 2m 13s
200:	learn: 0.4361810	total: 9.82s	remaining: 2m 41s
400:	learn: 0.4093772	total: 18.3s	remaining: 2m 21s
600:	learn: 0.3853065	total: 27.1s	remaining: 2m 10s
800:	learn: 0.3606725	total: 35.7s	remaining: 2m
1000:	learn: 0.3400807	total: 44.2s	remaining: 1m 50s
1200:	learn: 0.3213106	total: 52.8s	remaining: 1m 40s
1400:	learn: 0.3049037	total: 1m 1s	remaining: 1m 31s
1600:	learn: 0.2904229	total: 1m 9s	remaining: 1m 22s
1800:	learn: 0.2767792	total: 1m 18s	remaining: 1m 13s
2000:	learn: 0.2649746	total: 1m 26s	remaining: 1m 4s
2200:	learn: 0.2522839	total: 1m 34s	remaining: 56.1s
2400:	learn: 0.2399559	total: 1m 43s	remaining: 47.4s
2600:	learn: 0.2277564	total: 1m 52s	remaining: 38.8s
280

## 3. CAT Boost + Light GBM with oof

In [10]:
"""
BEST BLEND SCRIPT — Restaurant Turnover Prediction
--------------------------------------------------
Adds:
1) Frequency encoding (high-signal categoricals)
2) SAFE OOF target encoding (high-signal categoricals only, using log1p(target))
3) Diverse CatBoost ensemble (varied depth/seed/bagging/random_strength)
4) LightGBM model on same engineered data
5) Blend predictions: 0.75 * CatBoost + 0.25 * LightGBM

NOTE: City has a -1 value -> we convert categorical columns to string,
so "-1" becomes a valid category (no special case needed).
"""


import lightgbm as lgb


# =========================
# 1) SETTINGS
# =========================
TRAIN_PATH = "Train_dataset_.csv"
TEST_PATH  = "Test_dataset_.csv"

ID_COL     = "Registration Number"
TARGET_COL = "Annual Turnover"

SEED = 42

# OOF settings
N_SPLITS = 5
N_BINS   = 10

# Restrict encoding to high-signal columns only (this matters a lot)
OOF_FREQ_COLS = [
    "City",
    "Cuisine",
    "Restaurant Type",
    "Restaurant Theme",
    "Resturant Tier",
    "Restaurant City Tier",
    "Restaurant Location",
    # If you want to try 1 more, uncomment:
    # "Endorsed By",
]


# =========================
# 2) LOAD DATA
# =========================
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

if ID_COL not in train_df.columns or ID_COL not in test_df.columns:
    raise ValueError(f"'{ID_COL}' must exist in both train and test.")
if TARGET_COL not in train_df.columns:
    raise ValueError(f"'{TARGET_COL}' missing from train.")


# =========================
# 3) SPLIT X/y (DROP ID)
# =========================
y = train_df[TARGET_COL].copy()
y_log = np.log1p(y)

X = train_df.drop(columns=[TARGET_COL, ID_COL]).copy()
X_test = test_df.drop(columns=[ID_COL]).copy()

# Align test -> train columns (safety)
X_test = X_test.reindex(columns=X.columns, fill_value=np.nan)


# =========================
# 4) BASIC CLEANING + CATEGORICAL HANDLING
# =========================
# Identify categoricals by dtype
cat_cols_all = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols_all = [c for c in X.columns if c not in cat_cols_all]

# Force ALL categorical columns to string.
# This handles City=-1 (it becomes "-1" category) and keeps consistency.
for c in cat_cols_all:
    X[c] = X[c].astype(str).replace("nan", "Unknown").fillna("Unknown")
    X_test[c] = X_test[c].astype(str).replace("nan", "Unknown").fillna("Unknown")

# Numeric columns: coerce + median impute (train median applied to test)
for c in num_cols_all:
    X[c] = pd.to_numeric(X[c], errors="coerce")
    X_test[c] = pd.to_numeric(X_test[c], errors="coerce")
    med = X[c].median()
    X[c] = X[c].fillna(med)
    X_test[c] = X_test[c].fillna(med)


# =========================
# 5) KEEP ONLY EXISTING OOF/FREQ COLS
# =========================
OOF_FREQ_COLS = [c for c in OOF_FREQ_COLS if c in X.columns]
if not OOF_FREQ_COLS:
    raise ValueError("None of the specified OOF_FREQ_COLS exist in your dataset. Check column names.")

print("OOF/FREQ cols used:", OOF_FREQ_COLS)


# =========================
# 6) FREQUENCY ENCODING
# =========================
def add_frequency_encoding(X_df, X_test_df, cols):
    X_df = X_df.copy()
    X_test_df = X_test_df.copy()

    for c in cols:
        freq = X_df[c].value_counts(dropna=False) / len(X_df)
        X_df[f"{c}__freq"] = X_df[c].map(freq).fillna(0).astype("float32")
        X_test_df[f"{c}__freq"] = X_test_df[c].map(freq).fillna(0).astype("float32")
    return X_df, X_test_df

X_fe, X_test_fe = add_frequency_encoding(X, X_test, OOF_FREQ_COLS)


# =========================
# 7) SAFE OOF TARGET ENCODING (LOG TARGET)
# =========================
def make_bins(y_raw: pd.Series, n_bins=10):
    # bins based on raw target (not log) so distribution stratifies well
    return pd.qcut(y_raw, q=n_bins, duplicates="drop").astype(str)

def add_oof_target_encoding(X_df, X_test_df, y_log, cols, n_splits=5, n_bins=10, seed=42):
    """
    OOF mean target encoding:
    - Train: mean(log target) computed on folds excluding the row's fold
    - Test: mean(log target) computed on full train
    """
    X_df = X_df.copy()
    X_test_df = X_test_df.copy()

    y_raw = np.expm1(y_log)
    bins = make_bins(pd.Series(y_raw), n_bins=n_bins)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    global_mean = float(y_log.mean())

    for c in cols:
        X_df[f"{c}__oof_te"] = np.nan

    # OOF fill
    for tr_idx, val_idx in skf.split(X_df, bins):
        X_tr = X_df.iloc[tr_idx]
        y_tr = y_log.iloc[tr_idx]
        X_val = X_df.iloc[val_idx]

        for c in cols:
            mapping = y_tr.groupby(X_tr[c]).mean()
            X_df.loc[X_df.index[val_idx], f"{c}__oof_te"] = X_val[c].map(mapping)

    # Fill train NaNs, build test encodings from full mapping
    for c in cols:
        X_df[f"{c}__oof_te"] = X_df[f"{c}__oof_te"].fillna(global_mean).astype("float32")
        full_mapping = y_log.groupby(X_df[c]).mean()
        X_test_df[f"{c}__oof_te"] = X_test_df[c].map(full_mapping).fillna(global_mean).astype("float32")

    return X_df, X_test_df

X_fe, X_test_fe = add_oof_target_encoding(
    X_fe, X_test_fe, y_log, OOF_FREQ_COLS,
    n_splits=N_SPLITS, n_bins=N_BINS, seed=SEED
)

print("Final feature count after encodings:", X_fe.shape[1])


# =========================
# 8) CATBOOST ENSEMBLE (DIVERSE)
# =========================
# CatBoost will still treat the ORIGINAL categoricals natively.
cat_features_idx = [X_fe.columns.get_loc(c) for c in cat_cols_all]

cat_models = [
    # shallow + robust
    dict(iterations=4500, learning_rate=0.05,  depth=5,  l2_leaf_reg=10, subsample=0.75,
         bagging_temperature=1.0, random_strength=1.5, random_seed=SEED),
    dict(iterations=4500, learning_rate=0.05,  depth=5,  l2_leaf_reg=12, subsample=0.80,
         bagging_temperature=1.5, random_strength=2.0, random_seed=SEED+1),

    # medium
    dict(iterations=5500, learning_rate=0.04,  depth=7,  l2_leaf_reg=7,  subsample=0.80,
         bagging_temperature=0.8, random_strength=1.2, random_seed=SEED+2),
    dict(iterations=6000, learning_rate=0.04,  depth=8,  l2_leaf_reg=6,  subsample=0.85,
         bagging_temperature=0.7, random_strength=1.0, random_seed=SEED+3),

    # deeper but controlled
    dict(iterations=7000, learning_rate=0.035, depth=9,  l2_leaf_reg=8,  subsample=0.85,
         bagging_temperature=0.6, random_strength=1.3, random_seed=SEED+4),
    dict(iterations=7000, learning_rate=0.035, depth=10, l2_leaf_reg=9,  subsample=0.90,
         bagging_temperature=0.5, random_strength=1.5, random_seed=SEED+5),

    # extra diversity: different subsample + randomness
    dict(iterations=6500, learning_rate=0.037, depth=8,  l2_leaf_reg=9,  subsample=0.70,
         bagging_temperature=1.2, random_strength=2.5, random_seed=SEED+6),
    dict(iterations=6500, learning_rate=0.037, depth=6,  l2_leaf_reg=6,  subsample=0.95,
         bagging_temperature=0.9, random_strength=0.8, random_seed=SEED+7),
]

cat_preds_log = []
for i, p in enumerate(cat_models, 1):
    print(f"\n[CatBoost {i}/{len(cat_models)}] depth={p['depth']} lr={p['learning_rate']} sub={p['subsample']} seed={p['random_seed']}")
    m = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        verbose=200,
        allow_writing_files=False,
        **p
    )
    m.fit(X_fe, y_log, cat_features=cat_features_idx)
    cat_preds_log.append(m.predict(X_test_fe))

cat_ens_log = np.mean(np.vstack(cat_preds_log), axis=0)


# =========================
# 9) LIGHTGBM MODEL (ON SAME ENGINEERED DATA)
# =========================
# LightGBM needs numeric input. We'll encode original categoricals using pandas category codes.
# (We keep CatBoost handling categoricals natively; LGBM gets a numeric view.)

def to_lgb_matrix(X_df: pd.DataFrame, cat_cols):
    X_lgb = X_df.copy()
    for c in cat_cols:
        X_lgb[c] = X_lgb[c].astype("category").cat.codes.astype("int32")
    return X_lgb

X_lgb = to_lgb_matrix(X_fe, cat_cols_all)
X_test_lgb = to_lgb_matrix(X_test_fe, cat_cols_all)

lgb_params = dict(
    objective="regression",
    metric="rmse",
    learning_rate=0.03,
    num_leaves=63,
    feature_fraction=0.85,
    bagging_fraction=0.85,
    bagging_freq=1,
    min_data_in_leaf=40,
    lambda_l2=5.0,
    seed=SEED,
    verbosity=-1,
)

lgb_train = lgb.Dataset(X_lgb, label=y_log)
lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=4000
)

lgb_pred_log = lgb_model.predict(X_test_lgb)


# =========================
# 10) BLEND + SUBMISSION
# =========================
# Blend in LOG space (more stable), then invert
BLEND_CAT = 0.75
BLEND_LGB = 0.25

final_pred_log = (BLEND_CAT * cat_ens_log) + (BLEND_LGB * lgb_pred_log)

final_pred = np.expm1(final_pred_log)
final_pred = np.clip(final_pred, 0, None)

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL].values,
    TARGET_COL: final_pred
})

submission.to_csv("submission_best_blend.csv", index=False)
print("\n✔ Saved: submission_best_blend.csv")
print(submission.head())


Train shape: (3493, 34)
Test shape : (500, 33)
OOF/FREQ cols used: ['City', 'Cuisine', 'Restaurant Type', 'Restaurant Theme', 'Resturant Tier', 'Restaurant City Tier', 'Restaurant Location']
Final feature count after encodings: 46

[CatBoost 1/8] depth=5 lr=0.05 sub=0.75 seed=42
0:	learn: 0.5434279	total: 106ms	remaining: 7m 56s
200:	learn: 0.4522304	total: 9.65s	remaining: 3m 26s
400:	learn: 0.4311899	total: 21.3s	remaining: 3m 37s
600:	learn: 0.4135672	total: 32.8s	remaining: 3m 32s
800:	learn: 0.3967937	total: 44.2s	remaining: 3m 24s
1000:	learn: 0.3817339	total: 55.6s	remaining: 3m 14s
1200:	learn: 0.3682027	total: 1m 7s	remaining: 3m 6s
1400:	learn: 0.3564600	total: 1m 19s	remaining: 2m 56s
1600:	learn: 0.3450833	total: 1m 31s	remaining: 2m 46s
1800:	learn: 0.3340043	total: 1m 43s	remaining: 2m 34s
2000:	learn: 0.3223655	total: 1m 56s	remaining: 2m 25s
2200:	learn: 0.3128695	total: 2m 10s	remaining: 2m 15s
2400:	learn: 0.3044113	total: 2m 22s	remaining: 2m 4s
2600:	learn: 0.296581